In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"lfreedom2750","key":"fb46b035b65134128288a6ce5f370912"}'}

In [3]:
!pip install -q kaggle

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

!chmod 600 ~/.kaggle/kaggle.json

In [14]:
!kaggle datasets download -d lfreedom2750/mobilenetv3-data-fs
!unzip mobilenetv3-data-fs.zip

Dataset URL: https://www.kaggle.com/datasets/lfreedom2750/mobilenetv3-data-fs
License(s): unknown
mobilenetv3-data-fs.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  mobilenetv3-data-fs.zip
replace X_test_selected.npy? [y]es, [n]o, [A]ll, [N]one, [r]ename: A
  inflating: X_test_selected.npy     
  inflating: X_train_selected.npy    A
A

  inflating: X_valid_selected.npy    
  inflating: mobilenetv3_fs.pth      
  inflating: y_test.npy              
  inflating: y_train.npy             
  inflating: y_valid.npy             


In [2]:
import csv
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from PIL import Image

In [11]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class ReshapeResizeDataset(torch.utils.data.Dataset):
    def __init__(self, X_path, y_path, reshape_to=(3, 27, 30), resize_to=(224, 224)):
        self.X = np.load(X_path)
        self.y = np.load(y_path)
        self.reshape_to = reshape_to
        self.resize_to = resize_to
        self.flat_dim = np.prod(reshape_to)

        self.pad_width = self.flat_dim - self.X.shape[1]
        if self.pad_width < 0:
            raise ValueError("Vector length > reshape size!")

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]
        if self.pad_width > 0:
            x = np.pad(x, (0, self.pad_width), mode='constant')
        x = x.reshape(self.reshape_to)
        x = torch.tensor(x, dtype=torch.float32).unsqueeze(0)
        x = F.interpolate(x, size=self.resize_to, mode='bilinear')
        x = x.squeeze(0)
        y = self.y[idx]
        return x, int(y)

In [15]:
base_path = "/content"

train_dataset = ReshapeResizeDataset(f"{base_path}/X_train_selected.npy", f"{base_path}/y_train.npy")
val_dataset   = ReshapeResizeDataset(f"{base_path}/X_valid_selected.npy",   f"{base_path}/y_valid.npy")
test_dataset  = ReshapeResizeDataset(f"{base_path}/X_test_selected.npy",  f"{base_path}/y_test.npy")

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64,  shuffle=False,   num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64,  shuffle=False,   num_workers=2, pin_memory=True)

In [21]:
import torch.nn as nn
from torchvision import models

class MobileNetV3(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
        in_features = self.model.classifier[3].in_features
        self.model.classifier[3] = nn.Linear(in_features, 2)

    def forward(self, x):
        return self.model(x)

class MobileNetV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        num_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(num_features, 2)

    def forward(self, x):
        return self.model(x)

In [18]:
import torch
from collections import OrderedDict

def load_teacher_model(path, device="cuda"):
    model = MobileNetV3()
    state_dict = torch.load(path, map_location=device)

    new_state_dict = OrderedDict((k.replace("module.", ""), v) for k, v in state_dict.items())
    model.load_state_dict(new_state_dict)

    model.to(device)
    model.eval()
    return model

In [22]:
class DistillLoss(nn.Module):
    def __init__(self, T=4.0, alpha=0.7):
        super().__init__()
        self.T = T
        self.alpha = alpha
        self.kld = nn.KLDivLoss(reduction='batchmean')
        self.ce = nn.CrossEntropyLoss()

    def forward(self, student_logits, teacher_logits, true_labels):
        kd = self.kld(F.log_softmax(student_logits / self.T, dim=1),
                      F.softmax(teacher_logits / self.T, dim=1)) * (self.T ** 2)
        ce = self.ce(student_logits, true_labels)
        return self.alpha * kd + (1 - self.alpha) * ce

In [23]:
teacher = load_teacher_model("/content/mobilenetv3_fs.pth", device="cuda")
student = MobileNetV2()

if torch.cuda.device_count() > 1:
    print(f"Sử dụng {torch.cuda.device_count()} GPU với DataParallel.")
    student = nn.DataParallel(student)
    teacher = nn.DataParallel(teacher)

kd_loss_fn = DistillLoss(T=4.0, alpha=0.7)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth
100%|██████████| 9.83M/9.83M [00:00<00:00, 84.2MB/s]
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100%|██████████| 13.6M/13.6M [00:00<00:00, 103MB/s] 


In [ ]:
student = student.to(device)
teacher = teacher.to(device)

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score

def evaluate_model_on_validation(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = float('nan')

    return acc, auc

In [26]:
import time
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

def train_student_kd_with_validation(student, teacher, train_loader, val_loader, kd_loss_fn, optimizer, device, epochs=10):
    ce_loss_fn = nn.CrossEntropyLoss()
    start_training = time.time()

    for epoch in range(epochs):
        student.train()
        teacher.eval()

        total_kd_loss = 0
        total_student_loss = 0
        total_teacher_loss = 0

        all_probs = []
        all_labels = []

        start_epoch = time.time()

        progress_bar = tqdm(train_loader, desc=f"[Epoch {epoch+1}] Training", leave=False)

        for x, y in progress_bar:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()

            with torch.no_grad():
                t_logits = teacher(x)
                teacher_ce_loss = ce_loss_fn(t_logits, y)

            s_logits = student(x)

            student_ce_loss = ce_loss_fn(s_logits, y)
            kd_loss = kd_loss_fn(s_logits, t_logits, y)

            kd_loss.backward()
            optimizer.step()

            total_kd_loss += kd_loss.item()
            total_student_loss += student_ce_loss.item()
            total_teacher_loss += teacher_ce_loss.item()

            probs = torch.softmax(s_logits, dim=1)[:, 1].detach().cpu().numpy()
            all_probs.extend(probs)
            all_labels.extend(y.cpu().numpy())

            progress_bar.set_postfix({
                "KD": f"{kd_loss.item():.4f}",
                "StudentCE": f"{student_ce_loss.item():.4f}"
            })

        try:
            train_auc = roc_auc_score(all_labels, all_probs)
        except:
            train_auc = float('nan')

        epoch_time = time.time() - start_epoch

        val_acc, val_auc = evaluate_model_on_validation(student, val_loader, device)

        print(f"[Epoch {epoch+1}] "
              f"KD Loss: {total_kd_loss / len(train_loader):.4f} | "
              f"Student CE Loss: {total_student_loss / len(train_loader):.4f} | "
              f"Teacher CE Loss: {total_teacher_loss / len(train_loader):.4f} | "
              f"Train AUC: {train_auc:.4f} | "
              f"Val AUC: {val_auc:.4f} | Val ACC: {val_acc:.4f} | "
              f"Time: {epoch_time:.2f}s")

    total_time = time.time() - start_training
    print(f"⏱️ Tổng thời gian training: {total_time:.2f} giây ({total_time/60:.2f} phút)")

In [28]:
from sklearn.metrics import classification_report
import torch

def evaluate_models_report(student, teacher, dataloader, device):
    student.eval()
    teacher.eval()

    all_labels = []
    student_preds = []
    teacher_preds = []

    with torch.no_grad():
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            s_logits = student(x)
            t_logits = teacher(x)

            student_cls = torch.argmax(s_logits, dim=1).cpu().numpy()
            teacher_cls = torch.argmax(t_logits, dim=1).cpu().numpy()
            true_labels = y.cpu().numpy()

            student_preds.extend(student_cls)
            teacher_preds.extend(teacher_cls)
            all_labels.extend(true_labels)

    print("[Student Model] Classification Report:")
    print(classification_report(all_labels, student_preds, digits=4))

    print("[Teacher Model] Classification Report:")
    print(classification_report(all_labels, teacher_preds, digits=4))

In [ ]:
train_student_kd_with_validation(student, teacher, train_loader, val_loader, kd_loss_fn, optimizer, device, epochs=10)

[Epoch 1] KD Loss: 1.1940 | Student CE Loss: 0.5824 | Teacher CE Loss: 0.1303 | Train AUC: 0.8158 | Val AUC: 0.8294 | Val ACC: 0.7510 | Time: 311.18s


[Epoch 2] KD Loss: 0.7956 | Student CE Loss: 0.4164 | Teacher CE Loss: 0.1304 | Train AUC: 0.9041 | Val AUC: 0.8457 | Val ACC: 0.7661 | Time: 310.14s


[Epoch 3] KD Loss: 0.5653 | Student CE Loss: 0.2884 | Teacher CE Loss: 0.1303 | Train AUC: 0.9508 | Val AUC: 0.8488 | Val ACC: 0.7732 | Time: 310.11s


[Epoch 4] KD Loss: 0.4194 | Student CE Loss: 0.2081 | Teacher CE Loss: 0.1304 | Train AUC: 0.9745 | Val AUC: 0.8544 | Val ACC: 0.7725 | Time: 310.17s


[Epoch 5] KD Loss: 0.3330 | Student CE Loss: 0.1655 | Teacher CE Loss: 0.1304 | Train AUC: 0.9849 | Val AUC: 0.8542 | Val ACC: 0.7695 | Time: 310.00s


[Epoch 6] KD Loss: 0.2768 | Student CE Loss: 0.1414 | Teacher CE Loss: 0.1303 | Train AUC: 0.9900 | Val AUC: 0.8563 | Val ACC: 0.7750 | Time: 310.15s


[Epoch 7] KD Loss: 0.2397 | Student CE Loss: 0.1295 | Teacher CE Loss: 0.1304 | Train AUC: 0.9922 | Val AUC: 0.8536 | Val ACC: 0.7760 | Time: 310.10s


[Epoch 8] KD Loss: 0.2139 | Student CE Loss: 0.1237 | Teacher CE Loss: 0.1304 | Train AUC: 0.9932 | Val AUC: 0.8575 | Val ACC: 0.7749 | Time: 310.01s


[Epoch 9] KD Loss: 0.1931 | Student CE Loss: 0.1198 | Teacher CE Loss: 0.1304 | Train AUC: 0.9938 | Val AUC: 0.8551 | Val ACC: 0.7750 | Time: 309.89s


[Epoch 10] KD Loss: 0.1774 | Student CE Loss: 0.1173 | Teacher CE Loss: 0.1304 | Train AUC: 0.9942 | Val AUC: 0.8563 | Val ACC: 0.7774 | Time: 309.97s
⏱️ Tổng thời gian training: 3557.63 giây (59.29 phút)


In [29]:
student = MobileNetV2()
student_state_dict = torch.load("/content/mobilenetv2_fs.pth", map_location=device)

new_student_state_dict = OrderedDict((k.replace("module.", ""), v) for k, v in student_state_dict.items())
student.load_state_dict(new_student_state_dict)
teacher = load_teacher_model("/content/mobilenetv3_fs.pth", device="cuda")

student = student.to(device)
teacher = teacher.to(device)

evaluate_models_report(student, teacher, test_loader, device)

[Student Model] Classification Report:
              precision    recall  f1-score   support

           0     0.7738    0.7751    0.7744     20000
           1     0.7747    0.7734    0.7741     20000

    accuracy                         0.7742     40000
   macro avg     0.7743    0.7742    0.7742     40000
weighted avg     0.7743    0.7742    0.7742     40000

[Teacher Model] Classification Report:
              precision    recall  f1-score   support

           0     0.7200    0.7802    0.7489     20000
           1     0.7602    0.6966    0.7270     20000

    accuracy                         0.7384     40000
   macro avg     0.7401    0.7384    0.7379     40000
weighted avg     0.7401    0.7384    0.7379     40000

